# CLO tranche modeling (two-factor intuition)

**Start here:** This deep dive expands on `07_advanced_quant/correlation_and_credit_models.ipynb`; read the overview first for the common copula, latent-factor, and recovery vocabulary.

Use `LatentTwoFactor.clo_standard()` to map prepay/credit factors into the canonical finite-pool credit-loss simulation, then present subordination and expected tranche loss from the Rust-owned pool loss distribution.


## Concept

**Base correlation** is a market quoting convention: implied correlation is backed out per tranche so a model reproduces observed premiums. Economically, senior tranches are protected by subordination: they take losses only after junior attachment points are exhausted.

The CLO-standard two-factor model maps independent normals $(Z_1,Z_2)$ to the credit state $Z_c=L_{10}Z_1+L_{11}Z_2$. A name with asset correlation $\rho$ therefore uses portfolio-engine loadings $\sqrt{\rho}[L_{10},L_{11}]$. The Rust engine draws factors and idiosyncratic shocks, tests defaults, and returns the finite-pool loss distribution.

This notebook does not reprice a CLO. It keeps only attachment/detachment clipping as transparent presentation arithmetic over canonical pool losses.

## API walkthrough

- **`LatentTwoFactor.clo_standard()`** supplies the Cholesky coefficients for $Z_c$.
- **`CreditExposure.factor_loadings`** receives $\sqrt{\rho}[L_{10},L_{11}]$ for each homogeneous name.
- **`PortfolioLossConfig(..., CopulaSpec.gaussian())`** selects deterministic path count, seed, confidence, and copula.
- **`simulate_portfolio_loss`** returns path losses, expected loss, loss-positive VaR, and expected shortfall.
- A constant **`RecoverySpec`** supplies the pool LGD used on each exposure.

In [1]:
from finstack_quant.valuations.correlation import (
    CopulaSpec,
    CreditExposure,
    LatentTwoFactor,
    PortfolioLossConfig,
    RecoverySpec,
    simulate_portfolio_loss,
)

tf = LatentTwoFactor.clo_standard()
rec = RecoverySpec.constant(0.40).build()
print(f"CLO standard: {tf}")
print(f"Cholesky credit map: Z_c = {tf.cholesky_l10:.4f}*Z1 + {tf.cholesky_l11:.4f}*Z2")
print(f"Recovery model: {rec}")

CLO standard: LatentTwoFactor(prepay=0.1500, credit=0.3000, corr=-0.2000)
Cholesky credit map: Z_c = -0.2000*Z1 + 0.9798*Z2
Recovery model: RecoveryModel('Constant Recovery', expected=0.4000)


For each simulated pool loss fraction $\ell$, the presentation-layer tranche loss fraction between attachment $A$ and detachment $D$ is:

$$L^{\mathrm{tr}} = \frac{\min(\max(\ell - A, 0),\, D-A)}{D-A}$$

The default simulation and pool-loss aggregation are Rust-owned; this one clipping formula remains local because it is used once to explain the capital structure.

In [2]:
def tranche_loss_fraction(pool_loss: float, attach: float, detach: float) -> float:
    thickness = detach - attach
    penetration = min(max(pool_loss - attach, 0.0), thickness)
    return penetration / thickness


print(f"Example pool loss 8%: equity [0,3%] loss frac = {tranche_loss_fraction(0.08, 0.0, 0.03):.4f}")
print(f"Same pool: mezz [3,7%] loss frac = {tranche_loss_fraction(0.08, 0.03, 0.07):.4f}")
print(f"Same pool: senior [7,15%] loss frac = {tranche_loss_fraction(0.08, 0.07, 0.15):.4f}")

Example pool loss 8%: equity [0,3%] loss frac = 1.0000
Same pool: mezz [3,7%] loss frac = 1.0000
Same pool: senior [7,15%] loss frac = 0.1250


## Examples

Homogeneous pool: **200** names, PD **2.5%**, asset correlation **15%**, LGD from **40%** recovery, and **10,000** deterministic paths. Tranches are equity 0–3%, mezzanine 3–7%, senior 7–15%, and super-senior 15–100%.

Run the two-factor Gaussian finite-pool simulation, report pool EL/VaR/ES, then apply the single attachment/detachment clipping formula to each Rust-produced path loss.

In [3]:
n_pool = 200
pd_pool = 0.025
rho_asset = 0.15
lgd_pool = rec.lgd
name_notional = 1_000_000.0
total_notional = n_pool * name_notional
beta = rho_asset**0.5
credit_loadings = [beta * tf.cholesky_l10, beta * tf.cholesky_l11]

exposures = [
    CreditExposure(
        id=f"loan_{index + 1}",
        notional=name_notional,
        default_probability=pd_pool,
        lgd=lgd_pool,
        factor_loadings=credit_loadings,
    )
    for index in range(n_pool)
]
result = simulate_portfolio_loss(
    exposures,
    PortfolioLossConfig(10_000, 42, 0.99, CopulaSpec.gaussian()),
)
pool_losses = [loss / total_notional for loss in result.losses]

tranches = [
    ("equity", 0.0, 0.03),
    ("mezz", 0.03, 0.07),
    ("senior", 0.07, 0.15),
    ("super_senior", 0.15, 1.0),
]

print(f"Pool: n={n_pool}, PD={pd_pool}, rho={rho_asset}, LGD={lgd_pool:.2f}, loadings={credit_loadings}")
print(
    f"Pool EL={result.expected_loss / total_notional:.4%}, "
    f"VaR99={result.var / total_notional:.4%}, "
    f"ES99={result.expected_shortfall / total_notional:.4%}"
)
print("Expected tranche loss fraction:")
for label, attach, detach in tranches:
    expected_tranche_loss = sum(tranche_loss_fraction(pool_loss, attach, detach) for pool_loss in pool_losses) / len(
        pool_losses
    )
    print(f"  {label:<14} [{attach:.0%}, {detach:.0%}]  {expected_tranche_loss:.4%}")

Pool: n=200, PD=0.025, rho=0.15, LGD=0.60, loadings=[-0.07745966692414835, 0.3794733192202055]
Pool EL=1.4655%, VaR99=7.8000%, ES99=9.8079%
Expected tranche loss fraction:
  equity         [0%, 3%]  41.3300%
  mezz           [3%, 7%]  4.8900%
  senior         [7%, 15%]  0.3611%
  super_senior   [15%, 100%]  0.0013%


## Takeaways

- `LatentTwoFactor.clo_standard()` maps directly into two systematic exposure loadings $\sqrt{\rho}[L_{10},L_{11}]$.
- The canonical Rust engine owns finite-pool default simulation, deterministic RNG, loss aggregation, and pool VaR/ES.
- Subordination clips the same pool loss sequentially: equity is first-loss and super-senior is last-loss.
- Base correlation is a per-tranche implied parameter; one homogeneous asset correlation remains a pedagogical simplification.
- Pair this with `recovery_modeling.ipynb` for factor-dependent LGD.